In [ ]:
import random
from collections import deque

# Định nghĩa các hằng số dùng chung
GOAL = [[1, 2, 3], [4, 5, 6], [7, 8, 0]]
OPPOSITES = {'UP': 'DOWN', 'DOWN': 'UP', 'LEFT': 'RIGHT', 'RIGHT': 'LEFT'}

def is_solvable(state):
    """Đếm số lần đảo vị trí (inversions) gọn gàng bằng Generator để đảm bảo luôn giải được"""
    inversions = sum(
        1 for i in range(9) for j in range(i + 1, 9)
        if state[i] and state[j] and state[i] > state[j]
    )
    return inversions % 2 == 0

def generate_random_board():
    """Khởi tạo bàn cờ ngẫu nhiên hợp lệ"""
    while True:
        nums = list(range(9))
        random.shuffle(nums)
        if is_solvable(nums):
            return [nums[i:i + 3] for i in range(0, 9, 3)]

def find_blank(board):
    for r in range(3):
        for c in range(3):
            if board[r][c] == 0:
                return r, c

def get_legal_moves(board):
    r, c = find_blank(board)
    moves = {}
    if r > 0: moves['UP'] = (r - 1, c)
    if r < 2: moves['DOWN'] = (r + 1, c)
    if c > 0: moves['LEFT'] = (r, c - 1)
    if c < 2: moves['RIGHT'] = (r, c + 1)
    return moves

def apply_move(board, move):
    r, c = find_blank(board)
    nr, nc = get_legal_moves(board)[move]
    new_board = [row[:] for row in board]
    new_board[r][c], new_board[nr][nc] = new_board[nr][nc], new_board[r][c]
    return new_board


In [ ]:
def predict_score(board, visited_states):
    """[P1] Tính số ô sai vị trí. Cộng thêm điểm phạt cực nặng nếu trạng thái đã đi qua để tránh vòng lặp."""
    score = sum(
        1 for r in range(3) for c in range(3)
        if board[r][c] and board[r][c] != GOAL[r][c]
    )

    # Kỹ thuật tránh lặp: Phạt 100 điểm nếu AI định đi lại một trạng thái cũ
    if board in visited_states:
        score += 100

    return score

def print_board(board, title=None):
    """In ma trận cực ngắn gọn"""
    if title: 
        print(f"\n{title}")
    for row in board:
        print(" ".join(f"[{n if n else ' '}]" for n in row))

def board_to_tuple(board):
    """Chuyển list of list thành tuple of tuple để dùng trong set (phục vụ BFS/DFS)"""
    return tuple(tuple(row) for row in board)


In [ ]:
def solve_heuristic(start_board, max_steps=5000):
    board = [row[:] for row in start_board]
    history = []
    visited_states = [board]
    status, reason = "HALTED", "Quá số bước cho phép"

    print_board(board, "TRẠNG THÁI BẮT ĐẦU (Heuristic AI)")

    for step in range(1, max_steps + 1):
        if board == GOAL:
            status, reason = "SOLVED", "Tuyệt vời! Đã tìm thấy đích"
            break

        legal_moves = get_legal_moves(board)

        # [P4] Chống đi lùi trực tiếp
        if history:
            legal_moves.pop(OPPOSITES.get(history[-1]), None)

        # [P1] AI dự đoán nước đi tốt nhất
        best_move = min(
            legal_moves, 
            key=lambda m: predict_score(apply_move(board, m), visited_states)
        )

        history.append(best_move)
        board = apply_move(board, best_move)
        visited_states.append(board)
        
        # Đã comment print để tránh in ra quá dài
        # print_board(board, f"=> Bước {step}: Đi [{best_move}]")

    print("\nKẾT QUẢ HEURISTIC AI")
    print(f"Tổng số bước: {len(history)}")
    print(f"Trạng thái: {status} ({reason})")
    return history


## Áp dụng Thuật toán BFS (Breadth-First Search)
Duyệt theo chiều rộng, tìm được đường đi ngắn nhất.

In [ ]:
def solve_bfs(start_board):
    """Giải bằng Breadth-First Search (Duyệt theo chiều rộng)"""
    queue = deque([(start_board, [])])
    visited = set()
    visited.add(board_to_tuple(start_board))
    
    print_board(start_board, "TRẠNG THÁI BẮT ĐẦU (BFS)")
    nodes_explored = 0
    
    while queue:
        current_board, path = queue.popleft()
        nodes_explored += 1
        
        if current_board == GOAL:
            print("\nKẾT QUẢ BFS")
            print(f"Trạng thái: SOLVED (Đã tìm thấy đích)")
            print(f"Số bước ngắn nhất: {len(path)}")
            print(f"Số trạng thái đã duyệt: {nodes_explored}")
            return path
            
        legal_moves = get_legal_moves(current_board)
        
        for move in legal_moves:
            if path and move == OPPOSITES.get(path[-1]):
                continue
                
            new_board = apply_move(current_board, move)
            board_tuple = board_to_tuple(new_board)
            
            if board_tuple not in visited:
                visited.add(board_tuple)
                queue.append((new_board, path + [move]))
                
    print("\nKẾT QUẢ BFS: Không tìm thấy giải pháp!")
    return []


## Áp dụng Thuật toán DFS (Depth-First Search)
Duyệt theo chiều sâu. Có giới hạn độ sâu (Max Depth) để tránh vòng lặp vô tận hoặc tốn quá nhiều thời gian.

In [ ]:
def solve_dfs(start_board, max_depth=30):
    """Giải bằng Depth-First Search (Duyệt theo chiều sâu) - Có giới hạn độ sâu"""
    stack = [(start_board, [])]
    visited = {board_to_tuple(start_board): 0}
    
    print_board(start_board, f"TRẠNG THÁI BẮT ĐẦU (DFS - Max Depth: {max_depth})")
    nodes_explored = 0
    
    while stack:
        current_board, path = stack.pop()
        nodes_explored += 1
        current_depth = len(path)
        
        if current_board == GOAL:
            print("\nKẾT QUẢ DFS")
            print(f"Trạng thái: SOLVED (Đã tìm thấy đích)")
            print(f"Số bước: {len(path)}")
            print(f"Số trạng thái đã duyệt: {nodes_explored}")
            return path
            
        if current_depth >= max_depth:
            continue
            
        legal_moves = get_legal_moves(current_board)
        
        for move in legal_moves:
            if path and move == OPPOSITES.get(path[-1]):
                continue
                
            new_board = apply_move(current_board, move)
            board_tuple = board_to_tuple(new_board)
            
            if board_tuple not in visited or visited[board_tuple] > current_depth + 1:
                visited[board_tuple] = current_depth + 1
                stack.append((new_board, path + [move]))
                
    print(f"\nKẾT QUẢ DFS: HALTED (Không tìm thấy giải pháp trong {max_depth} bước)")
    return []


In [ ]:
# Khởi tạo chung 1 bàn cờ ngẫu nhiên
initial_board = generate_random_board()

print("=" * 50)
print("1. CHẠY THUẬT TOÁN HEURISTIC AI CŨ")
print("=" * 50)
solve_heuristic(initial_board)

print("\n" + "=" * 50)
print("2. CHẠY THUẬT TOÁN BFS (BREADTH-FIRST SEARCH)")
print("=" * 50)
solve_bfs(initial_board)

print("\n" + "=" * 50)
print("3. CHẠY THUẬT TOÁN DFS (DEPTH-FIRST SEARCH)")
print("=" * 50)
# DFS có thể duyệt rất sâu, đặt max_depth nhỏ (ví dụ: 25) để tránh tràn/chạy quá lâu
solve_dfs(initial_board, max_depth=25)
